<a href="https://colab.research.google.com/github/VickyW2366/Shors-optimisations/blob/main/Optimisation_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install qiskit
#!pip install qiskit-aer
import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.circuit.library import QFT, QFTGate
from qiskit.visualization import plot_histogram
from math import gcd
from fractions import Fraction
import numpy as np
from math import gcd
from sympy.ntheory import factorint
import random

def approximate_qft_3qubit(circuit, n, epsilon=0.1):

    """
    Approximate QFT that drops rotations below a threshold.
    For 3 qubits, we can drop the smallest rotations (pi/4 and pi/2)
    depending on desired precision.

    Optimization: Drop rotations with angle < epsilon * pi
    """

    # First qubit - keep only significant rotations
    circuit.h(0)
    # Keep R2 (pi/2) if 0.5 >= epsilon
    if 0.5 >= epsilon:
        circuit.cp(np.pi/2, 1, 0)
    # Drop R3 (pi/4) if 0.25 < epsilon (often dropped in AQFT)
    if 0.25 >= epsilon:
        circuit.cp(np.pi/4, 2, 0)

    # Second qubit
    circuit.h(1)
    # Keep R2 if significant
    if 0.5 >= epsilon:
        circuit.cp(np.pi/2, 2, 1)
    # Third qubit
    circuit.h(2)
    circuit.swap(0, 2)

    # Calculate gate reduction
    original_gates = 6  # 3 H + 3 CP
    optimized_gates = 3 + sum([1 if 0.5 >= epsilon else 0,
                                1 if 0.25 >= epsilon else 0,
                                1 if 0.5 >= epsilon else 0])
    print(f"Gate reduction: {original_gates} → {optimized_gates} gates")

# Calls the function to execute it
qc = QuantumCircuit(3)
approximate_qft_3qubit(qc, 3)
print(qc.draw())

Gate reduction: 6 → 6 gates
     ┌───┐                                        
q_0: ┤ H ├─■────────■───────────────────────────X─
     └───┘ │P(π/2)  │       ┌───┐               │ 
q_1: ──────■────────┼───────┤ H ├─■─────────────┼─
                    │P(π/4) └───┘ │P(π/2) ┌───┐ │ 
q_2: ───────────────■─────────────■───────┤ H ├─X─
                                          └───┘   
